### Fase 1: Exploración y Limpieza
- **Exploración Inicial:**

In [1]:
# Tratamiento de datos
# -----------------------------------------------------------------------
import pandas as pd # para el manejo de DataFrames
import numpy as np # para el manejo de arrays y operaciones matemáticas

# Visualización
# ------------------------------------------------------------------------------
import matplotlib.pyplot as plt # para crear gráficos
import seaborn as sns

# Configuración
# -----------------------------------------------------------------------
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames


import warnings
warnings.filterwarnings("ignore") # para evitar los mensajes de advertencia


In [2]:
df_vuelos = pd.read_csv("files/Customer Flight Activity.csv") # para cargar los datos de vuelo del cliente
df_hclientes= pd.read_csv("files/Customer Loyalty History.csv") # para cargar los datos del historial de lealtad del cliente

## Vamos a analizar en df_vuelos

In [36]:
df_vuelos.head() # para visualizar las primeras filas del DataFrame vuelos y tener una idea de su estructura

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
0,100018,2017,1,3,0,3,1521,152.0,0,0
1,100102,2017,1,10,4,14,2030,203.0,0,0
2,100140,2017,1,6,0,6,1200,120.0,0,0
3,100214,2017,1,0,0,0,0,0.0,0,0
4,100272,2017,1,0,0,0,0,0.0,0,0


In [37]:
df_vuelos.shape # para conocer el número de filas y columnas del DataFrame vuelos

(403760, 10)

In [38]:
df_vuelos.info() # para conocer el tipo de datos de cada columna del DataFrame vuelos y detectar posibles problemas de formato
#Vemos que en este dataframe no hay valores nulos ya que todas las columnas tienen 405624 valores(que es el total de filas del df(como hemos visto en el shape) 

<class 'pandas.DataFrame'>
Index: 403760 entries, 0 to 405623
Data columns (total 10 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Loyalty Number               403760 non-null  int64  
 1   Year                         403760 non-null  int64  
 2   Month                        403760 non-null  int64  
 3   Flights Booked               403760 non-null  int64  
 4   Flights with Companions      403760 non-null  int64  
 5   Total Flights                403760 non-null  int64  
 6   Distance                     403760 non-null  int64  
 7   Points Accumulated           403760 non-null  float64
 8   Points Redeemed              403760 non-null  int64  
 9   Dollar Cost Points Redeemed  403760 non-null  int64  
dtypes: float64(1), int64(9)
memory usage: 33.9 MB


A continuación,vamos a revisar si encontramos algún dato que debamos cambiarle el tipo:

- Veo que sería interesante cambiar a tipo fecha las columnas "Year" y "Month"
- Eliminar espacios de los nombres de las columnas y poner "-" (buenas prácticas)
- He revisado y es correcto que el tipo de la columna Points Acumulated sea tipo float ya que hay decimales

In [39]:
#Vamos a ver si hay valores duplicados en el DataFrame vuelos
df_vuelos.duplicated().sum() # Vemos que hay 1864 filas duplicadas que eliminaremos antes de hacer la unión de los dataframes para evitar que se dupliquen datos al hacer la unión

np.int64(0)

In [40]:
# Eliminamos las filas duplicadas del DataFrame vuelos
df_vuelos = df_vuelos.drop_duplicates() 

In [41]:
# Verificamos que se han eliminado las filas duplicadas
df_vuelos.duplicated().sum() # Vemos que efectivamente ya no hay filas duplicadas

np.int64(0)

In [42]:
#Vamos a comprobar que tal y como hemos visto en el info, no hay valores nulos
df_vuelos.isna().sum()/df_vuelos.shape[0]*100 #lo ponemos en porcentaje para verlos en valores relativos

Loyalty Number                 0.0
Year                           0.0
Month                          0.0
Flights Booked                 0.0
Flights with Companions        0.0
Total Flights                  0.0
Distance                       0.0
Points Accumulated             0.0
Points Redeemed                0.0
Dollar Cost Points Redeemed    0.0
dtype: float64

In [43]:
df_vuelos.columns

Index(['Loyalty Number', 'Year', 'Month', 'Flights Booked',
       'Flights with Companions', 'Total Flights', 'Distance',
       'Points Accumulated', 'Points Redeemed', 'Dollar Cost Points Redeemed'],
      dtype='str')

In [44]:
#Vamos a comprobar que la columna Loyalty Number a pesar de ser un identificador único. en el df_vuloes SI tiene valores duplicados
df_vuelos["Loyalty Number"].duplicated().sum()

#Un mismo cliente puede tener actividad en varios meses, por eso su Loyalty Number aparece varias veces (es lógico por lo que no debemos eliminar los duplicados)
##He revisado y las demás columnas es normal que tengan valores duplicados (Los años se repiten, los puntos pueden coincidir, etc)

np.int64(387023)

In [45]:
df_vuelos.describe().round(2) # para obtener estadísticas descriptivas de las columnas numéricas del DataFrame vuelos y tener una idea de su distribución y posibles valores atípicos

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
count,403760.00,403760.0,403760.00,403760.00,403760.00,403760.00,403760.00,403760.00,403760.00,403760.00
mean,549875.38,2017.5,6.50,4.13,1.04,5.17,1214.46,124.26,30.84,2.50
std,258961.51,0.5,3.45,5.23,2.08,6.53,1434.10,146.70,125.76,10.17
min,100018.00,2017.0,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,326699.00,2017.0,4.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,550598.00,2018.0,7.00,1.00,0.00,1.00,525.00,53.00,0.00,0.00
75%,772152.00,2018.0,10.00,8.00,1.00,10.00,2342.00,240.00,0.00,0.00
max,999986.00,2018.0,12.00,21.00,11.00,32.00,6293.00,676.50,876.00,71.00


In [69]:
df_vuelos.describe(include=np.number).round(2).T # para obtener estadísticas descriptivas de las columnas numéricas
#En el info()ya hemos vist que no hay ninguna columna de tipo objectpor eso ya no es necesario analizarlo

,count,mean,std,min,25%,50%,75%,max
Loyalty Number,403760.0,549875.38,258961.51,100018.0,326699.0,550598.0,772152.0,999986.0
Year,403760.0,2017.50,0.50,2017.0,2017.0,2018.0,2018.0,2018.0
Month,403760.0,6.50,3.45,1.0,4.0,7.0,10.0,12.0
Flights Booked,403760.0,4.13,5.23,0.0,0.0,1.0,8.0,21.0
Flights with Companions,403760.0,1.04,2.08,0.0,0.0,0.0,1.0,11.0
Total Flights,403760.0,5.17,6.53,0.0,0.0,1.0,10.0,32.0
Distance,403760.0,1214.46,1434.10,0.0,0.0,525.0,2342.0,6293.0
Points Accumulated,403760.0,124.26,146.70,0.0,0.0,53.0,240.0,676.5
Points Redeemed,403760.0,30.84,125.76,0.0,0.0,0.0,0.0,876.0
Dollar Cost Points Redeemed,403760.0,2.50,10.17,0.0,0.0,0.0,0.0,71.0


## Vamos a analizar en df_hclientes:

#Realizamos los mismos pasos que ya hemos realizado en el df_vuelos

In [48]:
df_hclientes.shape

(16737, 16)

In [50]:
df_hclientes.columns

Index(['Loyalty Number', 'Country', 'Province', 'City', 'Postal Code',
       'Gender', 'Education', 'Salary', 'Marital Status', 'Loyalty Card',
       'CLV', 'Enrollment Type', 'Enrollment Year', 'Enrollment Month',
       'Cancellation Year', 'Cancellation Month'],
      dtype='str')

In [ ]:
df_hclientes.info()
#Vemos a simple vista que hay 3 columnas que tienen valores nulos (Cancellatio year, Cancellation month y salary)
#Más adelante analaizaremos los valores nulos y veremos que hacemos con ellos (Lo trataremos en la fase 2 Limpieza de datos) 

<class 'pandas.DataFrame'>
RangeIndex: 16737 entries, 0 to 16736
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Loyalty Number      16737 non-null  int64  
 1   Country             16737 non-null  str    
 2   Province            16737 non-null  str    
 3   City                16737 non-null  str    
 4   Postal Code         16737 non-null  str    
 5   Gender              16737 non-null  str    
 6   Education           16737 non-null  str    
 7   Salary              12499 non-null  float64
 8   Marital Status      16737 non-null  str    
 9   Loyalty Card        16737 non-null  str    
 10  CLV                 16737 non-null  float64
 11  Enrollment Type     16737 non-null  str    
 12  Enrollment Year     16737 non-null  int64  
 13  Enrollment Month    16737 non-null  int64  
 14  Cancellation Year   2067 non-null   float64
 15  Cancellation Month  2067 non-null   float64
dtypes: float64(4), 

In [ ]:
#Las siguientes columnas sería interesante cambiarle el tipo de dato una vez gestionados los nulos a fecha: Cancellation year, Cancellation month.
# Cambiar también el tipo de dato de Enrollment Year y Enrollment Month a fecha

In [ ]:
df_hclientes.duplicated().sum()
#No hay filas duplicadas 

np.int64(0)

In [ ]:
df_hclientes["Loyalty Number"].duplicated().sum()

#Vemos que en este dataframe el Loyalty number no está duplicado porque cada fila representa a un cliente diferente

np.int64(0)

In [ ]:
round(df_hclientes.isna().sum()/df_hclientes.shape[0]*100,2)#lo ponemos en porcentaje para verlos en valores relativos 
#Vemos que tal y como habíamos visto en el info, las columnas con valores nulos son Cancellation year, Cancellation month y Salary.
# En la fase 2 , los gestionaremos y veremos si los imputamos, si elimnamos columnas porque no nos aportan información relevante...

Loyalty Number         0.00
Country                0.00
Province               0.00
City                   0.00
Postal Code            0.00
Gender                 0.00
Education              0.00
Salary                25.32
Marital Status         0.00
Loyalty Card           0.00
CLV                    0.00
Enrollment Type        0.00
Enrollment Year        0.00
Enrollment Month       0.00
Cancellation Year     87.65
Cancellation Month    87.65
dtype: float64

In [ ]:
df_hclientes.describe().round(2).T
#Aquí,vemos hay valores negativos en la columna Salary y no tiene sentido,habrá que ver que error es.

,count,mean,std,min,25%,50%,75%,max
Loyalty Number,16737.0,549735.88,258912.13,100018.00,326603.00,550434.00,772019.00,999986.00
Salary,12499.0,79245.61,35008.30,-58486.00,59246.50,73455.00,88517.50,407228.00
CLV,16737.0,7988.90,6860.98,1898.01,3980.84,5780.18,8940.58,83325.38
Enrollment Year,16737.0,2015.25,1.98,2012.00,2014.00,2015.00,2017.00,2018.00
Enrollment Month,16737.0,6.67,3.40,1.00,4.00,7.00,10.00,12.00
Cancellation Year,2067.0,2016.50,1.38,2013.00,2016.00,2017.00,2018.00,2018.00
Cancellation Month,2067.0,6.96,3.46,1.00,4.00,7.00,10.00,12.00


In [ ]:
df_hclientes.describe(include=str).round(2).T
#Vemos que la columna Country solamente tiene un valor único,por lo que es constante y no nos aporta información relevante
#Seguramente es candidata a ser eliminada (lo revisaremos en la fase 2)

,count,unique,top,freq
Country,16737,1,Canada,16737
Province,16737,11,Ontario,5404
City,16737,29,Toronto,3351
Postal Code,16737,55,V6E 3D9,911
Gender,16737,2,Female,8410
Education,16737,5,Bachelor,10475
Marital Status,16737,3,Married,9735
Loyalty Card,16737,3,Star,7637
Enrollment Type,16737,2,Standard,15766


In [85]:
df_hclientes["Country"].value_counts() # para confirmar que la columna Country tiene un único valor y es constante


Country
Canada    16737
Name: count, dtype: int64

In [84]:
df_hclientes["Country"].nunique()

1

Vamos a revisar los valores de las columnas que tienen pocos valores únicos para ver que contienen (ver cuales son los valores)

In [87]:
df_hclientes["Loyalty Card"].value_counts()

Loyalty Card
Star      7637
Nova      5671
Aurora    3429
Name: count, dtype: int64

In [88]:
df_hclientes["Enrollment Type"].value_counts()


Enrollment Type
Standard          15766
2018 Promotion      971
Name: count, dtype: int64

In [89]:
df_hclientes["Marital Status"].value_counts()

Marital Status
Married     9735
Single      4484
Divorced    2518
Name: count, dtype: int64

In [90]:
df_hclientes["Education"].value_counts()

Education
Bachelor                10475
College                  4238
High School or Below      782
Doctor                    734
Master                    508
Name: count, dtype: int64

## Vamos a hacer la unión de los dos dataframes anteriores

In [ ]:
#Para unir los dos dataframes, se va a usar la columna Loyalty Number que es el identificador único de cada cliente y que aparece en ambos dataframes. 
#La unión la realizaré usando un left merge ya que quiero conservar la información de los clientes que aún no han comprado ni reservado ningún vuelo,


In [91]:

df_final = pd.merge(df_hclientes, df_vuelos, on="Loyalty Number", how="left") 
  #El dataframe izquierdo es el que quieres conservar completo, aunque no haya coincidencias en el otro dataframe.
  # Si pusiéramos en el left el otro dataframe, solo se conservarían los clientes que tienen actividad.